# Session 8 — Trainable Quantum Transformer

Objective:
Replace random Q/K/V parameters with shared trainable parameters and evaluate whether learned quantum representations improve transformer performance.

In [1]:
import numpy as np

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

In [2]:
X, y = make_moons(
    n_samples=400,
    noise=0.15,
    random_state=42
)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

Train Shape: (320, 2)
Test Shape : (80, 2)


## Trainable Query, Key and Value Circuits

In [3]:
x_params = ParameterVector("x", 2)

q_theta = ParameterVector("q", 2)
k_theta = ParameterVector("k", 2)
v_theta = ParameterVector("v", 2)


def create_qkv_circuit(weight_params):

    qc = QuantumCircuit(2)

    qc.ry(x_params[0], 0)
    qc.ry(x_params[1], 1)

    qc.ry(weight_params[0], 0)
    qc.ry(weight_params[1], 1)

    qc.cx(0, 1)

    return qc


query_circuit = create_qkv_circuit(q_theta)
key_circuit = create_qkv_circuit(k_theta)
value_circuit = create_qkv_circuit(v_theta)

print("Trainable Q/K/V circuits created.")

Trainable Q/K/V circuits created.


In [4]:
q_weights = np.random.uniform(
    0,
    np.pi,
    2
)

k_weights = np.random.uniform(
    0,
    np.pi,
    2
)

v_weights = np.random.uniform(
    0,
    np.pi,
    2
)

print("Q Weights:", q_weights)
print("K Weights:", k_weights)
print("V Weights:", v_weights)

Q Weights: [2.09693466 0.54968662]
K Weights: [1.37351662 1.92431559]
V Weights: [2.48492247 1.73189026]


## Quantum Transformer Features

In [5]:
def transformer_embedding(sample):

    query_state = Statevector.from_instruction(
        query_circuit.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            q_theta[0]: q_weights[0],
            q_theta[1]: q_weights[1]
        })
    )

    key_state = Statevector.from_instruction(
        key_circuit.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            k_theta[0]: k_weights[0],
            k_theta[1]: k_weights[1]
        })
    )

    value_state = Statevector.from_instruction(
        value_circuit.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            v_theta[0]: v_weights[0],
            v_theta[1]: v_weights[1]
        })
    )

    attention_score = np.abs(
        np.vdot(
            query_state.data,
            key_state.data
        )
    )

    transformer_feature = (
        attention_score *
        np.abs(value_state.data)
    )

    return transformer_feature

## Training and Testing Transformer Embeddings

In [6]:
X_train_transformer = np.array([
    transformer_embedding(sample)
    for sample in X_train
])

X_test_transformer = np.array([
    transformer_embedding(sample)
    for sample in X_test
])

print("Train Transformer Shape:")
print(X_train_transformer.shape)

print("\nTest Transformer Shape:")
print(X_test_transformer.shape)

Train Transformer Shape:
(320, 4)

Test Transformer Shape:
(80, 4)


## Train Classification Head

In [8]:
classifier = LogisticRegression(
    max_iter=1000
)

classifier.fit(
    X_train_transformer,
    y_train
)

print("Classifier trained.")

Classifier trained.


## Evaluate Trainable Quantum Transformer

In [10]:
train_accuracy = classifier.score(
    X_train_transformer,
    y_train
)

test_accuracy = classifier.score(
    X_test_transformer,
    y_test
)

print("Train Accuracy:")
print(train_accuracy)

print("\nTest Accuracy:")
print(test_accuracy)

Train Accuracy:
0.859375

Test Accuracy:
0.8375


## Comparision with Session 7

In [11]:
session7_accuracy = 0.825

print("Session 7 Accuracy :", session7_accuracy)
print("Session 8 Accuracy :", test_accuracy)

difference = test_accuracy - session7_accuracy

print("\nDifference:")
print(difference)

Session 7 Accuracy : 0.825
Session 8 Accuracy : 0.8375

Difference:
0.012500000000000067
